In [1]:
from langchain_milvus import Milvus
from langchain_community.document_loaders import CSVLoader
from dotenv import load_dotenv
import os

load_dotenv()

/tmp/ipykernel_123728/3041212266.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


True

In [2]:
endpoint = os.environ["MILVUS_ENDPOINT"]
endpoint_token = os.environ["MILVUS_TOKEN"]

In [3]:
csv_path = "/home/gaurav/Documents/advanced_rag/data/Cleaned_Data.csv"

In [4]:
import pandas as pd

df = pd.read_csv(csv_path)

In [5]:
# df.dropna().to_csv(
#     "/home/gaurav/Documents/advanced_rag/data/Cleaned_Data.csv", index=False
# )

In [6]:
metadata_col = pd.read_csv(csv_path).drop(columns="Movie").columns.tolist()

In [7]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

csv_docs = CSVLoader(
    file_path=csv_path, metadata_columns=metadata_col, content_columns="Movie"
).load_and_split(
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
)


In [8]:
from langchain_openai import OpenAIEmbeddings

In [9]:
embedding_model = OpenAIEmbeddings()

In [10]:
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.schema import AttributeInfo

In [ ]:
# metadata_col_dict = df.drop(columns="Movie").dtypes.to_dict()

In [ ]:
metadata_col_dict

In [11]:
metadata_field_info = [
    AttributeInfo(
        name="LeadStudio",
        description="The film studio responsible for producing or distributing the movie.",
        type="string",
    ),
    AttributeInfo(
        name="RottenTomatoes",
        description="The movie's Rotten Tomatoes critic score.",
        type="integer",
    ),
    AttributeInfo(
        name="AudienceScore",
        description="The movie's audience rating score.",
        type="integer",
    ),
    AttributeInfo(
        name="Story",
        description="A brief description or summary of the movie's storyline.",
        type="string",
    ),
    AttributeInfo(
        name="Genre",
        description="The genre or genres of the movie, such as Comedy, Drama, or Action.",
        type="string",
    ),
    AttributeInfo(
        name="TheatersOpenWeek",
        description="The number of theaters in which the movie opened during its opening week.",
        type="integer",
    ),
    AttributeInfo(
        name="OpeningWeekend",
        description="The movie's opening weekend box-office revenue.",
        type="float",
    ),
    AttributeInfo(
        name="BOAvgOpenWeekend",
        description="The average box-office revenue per theater during the opening weekend.",
        type="integer",
    ),
    AttributeInfo(
        name="DomesticGross",
        description="The movie's total domestic box-office revenue.",
        type="float",
    ),
    AttributeInfo(
        name="ForeignGross",
        description="The movie's total international box-office revenue outside its domestic market.",
        type="float",
    ),
    AttributeInfo(
        name="WorldGross",
        description="The movie's total worldwide box-office revenue.",
        type="float",
    ),
    AttributeInfo(
        name="Budget",
        description="The total production budget of the movie.",
        type="integer",
    ),
    AttributeInfo(
        name="Profitability",
        description="The movie's profitability ratio, as represented in the dataset.",
        type="float",
    ),
    AttributeInfo(
        name="OpenProfit",
        description="The movie's opening-weekend profit metric, as defined in the dataset.",
        type="float",
    ),
    AttributeInfo(
        name="Year",
        description="The year the movie was released.",
        type="integer",
    ),
]


In [12]:
from advanced_rag.models import get_groq

groq_llm = get_groq()

In [13]:
# document_content_description = "Movies Names with their respective box office openings, rating, pprofitabily and other"
# from langchain_classic.retrievers.self_query.milvus import MilvusTranslator

# self_retriever = SelfQueryRetriever.from_llm(
#     llm=groq_llm,
#     vectorstore=metadata_db,
#     document_contents=document_content_description,
#     metadata_field_info=metadata_field_info,
#     enable_limit=True,
#     verbose=True,
#     structured_query_translator=MilvusTranslator(),
# )


In [ ]:
# query = "Find movies released after 2010"

# docs = self_retriever.invoke(query)

# print(f"Retrieved {len(docs)} documents")

# for i, doc in enumerate(docs, 1):
#     print(f"\n--- Document {i} ---")
#     print("Content:", doc.page_content[:300])
#     print("Metadata:", doc.metadata)

In [ ]:
from pymilvus import MilvusClient

In [ ]:
client = MilvusClient(uri=endpoint, token=endpoint_token)

In [ ]:
client.list_collections()

In [ ]:
laoded_schema = client.describe_collection("movies_metadata_collection")

In [ ]:
# client.drop_collection("movies_metadata_collection")

In [14]:
from pymilvus import DataType

metadata_schema = {
    "LeadStudio": {
        "dtype": DataType.VARCHAR,
        "max_length": 512,
    },
    "RottenTomatoes": {
        "dtype": DataType.INT64,
    },
    "AudienceScore": {
        "dtype": DataType.INT64,
    },
    "Story": {
        "dtype": DataType.VARCHAR,
        "max_length": 65535,
    },
    "Genre": {
        "dtype": DataType.VARCHAR,
        "max_length": 512,
    },
    "TheatersOpenWeek": {
        "dtype": DataType.INT64,
    },
    "OpeningWeekend": {
        "dtype": DataType.DOUBLE,
    },
    "BOAvgOpenWeekend": {
        "dtype": DataType.INT64,
    },
    "DomesticGross": {
        "dtype": DataType.DOUBLE,
    },
    "ForeignGross": {
        "dtype": DataType.DOUBLE,
    },
    "WorldGross": {
        "dtype": DataType.DOUBLE,
    },
    "Budget": {
        "dtype": DataType.INT64,
    },
    "Profitability": {
        "dtype": DataType.DOUBLE,
    },
    "OpenProfit": {
        "dtype": DataType.DOUBLE,
    },
    "Year": {
        "dtype": DataType.INT64,
    },
}


In [24]:
import pandas as pd

integer_fields = {
    "row",
    "RottenTomatoes",
    "AudienceScore",
    "TheatersOpenWeek",
    "BOAvgOpenWeekend",
    "Year",
}

float_fields = {
    "OpeningWeekend",
    "DomesticGross",
    "ForeignGross",
    "WorldGross",
    "Budget",
    "Profitability",
    "OpenProfit",
}


def convert_metadata(metadata):
    converted = metadata.copy()

    for key in integer_fields:
        if key in converted:
            value = converted[key]
            if value is not None and str(value).strip():
                number = float(value)
                if not number.is_integer():
                    raise ValueError(f"{key} is not a whole number: {value}")
                converted[key] = int(number)

    for key in float_fields:
        if key in converted:
            value = converted[key]
            if value is not None and str(value).strip():
                converted[key] = float(value)

    return converted

In [25]:
from langchain_core.documents import Document

typed_docs = [
    Document(
        page_content=doc.page_content,
        metadata=convert_metadata(doc.metadata),
    )
    for doc in csv_docs
]

In [28]:
metadata_schema["Budget"] = {
    "dtype": DataType.DOUBLE,
}

In [29]:
metadata_db = Milvus(
    embedding_function=embedding_model,
    collection_name="movies_metadata_collection_v5",
    connection_args={"uri": endpoint, "token": endpoint_token},
    metadata_schema=metadata_schema,
)

In [30]:
insertion_ids = metadata_db.add_documents(typed_docs)

/home/gaurav/Documents/advanced_rag/.venv/lib/python3.14/site-packages/langchain_milvus/vectorstores/milvus.py:1408: UserWarning: No ids provided and auto_id is False. Setting auto_id to True automatically.
  warnings.warn(


In [33]:
# len(insertion_ids)

In [39]:
from langchain_classic.chains.query_constructor.schema import AttributeInfo

metadata_field_info = [
    AttributeInfo(
        name="LeadStudio",
        description="The film studio responsible for producing or distributing the movie.",
        type="string",
    ),
    AttributeInfo(
        name="RottenTomatoes",
        description="The movie's Rotten Tomatoes critic score, on the dataset's rating scale.",
        type="integer",
    ),
    AttributeInfo(
        name="AudienceScore",
        description="The movie's audience rating score, on the dataset's rating scale.",
        type="integer",
    ),
    AttributeInfo(
        name="Story",
        description="A brief description or summary of the movie's storyline.",
        type="string",
    ),
    AttributeInfo(
        name="Genre",
        description="The movie's genre, such as Comedy, Drama, or Action.",
        type="string",
    ),
    AttributeInfo(
        name="TheatersOpenWeek",
        description="The number of theaters in which the movie opened during its opening week.",
        type="integer",
    ),
    AttributeInfo(
        name="OpeningWeekend",
        description="The movie's opening weekend box-office revenue.",
        type="float",
    ),
    AttributeInfo(
        name="BOAvgOpenWeekend",
        description="Average opening-weekend box-office revenue per theater.",
        type="integer",
    ),
    AttributeInfo(
        name="DomesticGross",
        description="The movie's total domestic box-office revenue.",
        type="float",
    ),
    AttributeInfo(
        name="ForeignGross",
        description="The movie's total international box-office revenue outside its domestic market.",
        type="float",
    ),
    AttributeInfo(
        name="Profitability",
        description="The movie's profitability ratio, as represented in the dataset.",
        type="float",
    ),
    AttributeInfo(
        name="OpenProfit",
        description="The movie's opening-weekend profit metric, as defined in the dataset.",
        type="float",
    ),
    AttributeInfo(
        name="Year",
        description="The year the movie was released.",
        type="integer",
    ),
    AttributeInfo(
        name="Budget",
        description=(
            "Movie production budget in millions of dollars. "
            "Values are stored in millions, not raw dollars."
        ),
        type="float",
    ),
    AttributeInfo(
        name="WorldGross",
        description=(
            "Total worldwide box-office revenue in millions of dollars. "
            "Values are stored in millions, not raw dollars."
        ),
        type="float",
    ),
]

In [40]:
document_content_description = """
A dataset of movies containing their titles and metadata about genre, lead studio,
release year, Rotten Tomatoes critic scores, audience scores, opening-weekend
box-office performance, number of theaters during opening week, domestic and
international gross revenue, worldwide gross revenue, production budget,
profitability, and opening-weekend profitability.

Financial values such as OpeningWeekend, DomesticGross, ForeignGross,
WorldGross, and Budget are stored in millions of dollars, if applicable to
the dataset. Ratings are numerical scores, Year represents the release year,
and Profitability and OpenProfit represent the dataset's profitability metrics.
Use the metadata fields to apply numerical comparisons, ranges, and categorical
filters when answering movie-related queries.
"""

In [41]:
from langchain_classic.retrievers.self_query.milvus import MilvusTranslator

self_retriever = SelfQueryRetriever.from_llm(
    llm=groq_llm,
    vectorstore=metadata_db,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    enable_limit=True,
    verbose=True,
    structured_query_translator=MilvusTranslator(),
)

In [45]:
query = """Find Action, Adventure, or Sci-Fi movies released between 2005 and 2018, inclusive. The Rotten Tomatoes score must be greater than 75, and the AudienceScore must be at least 80. The production budget must be below 150 million dollars, while worldwide gross revenue must exceed 400 million dollars and foreign gross revenue must be greater than domestic gross revenue. Profitability must be above 3.0. Only consider movies released in at least 3,000 theaters during their opening week. Prefer movies with the highest worldwide gross, followed by the highest Rotten Tomatoes score. Return at most 10 movies."""

In [47]:
docs = self_retriever.invoke(query)

print(f"Retrieved {len(docs)} documents")

for i, doc in enumerate(docs, 1):
    print(f"\n--- Document {i} ---")
    print("Content:", doc.page_content[:300])
    print("Metadata:", doc.metadata)

Retrieved 4 documents

--- Document 1 ---
Content: Movie: Mission Impossible 4
Metadata: {'BOAvgOpenWeekend': 8672, 'OpeningWeekend': 29.55, 'Year': 2011, 'DomesticGross': 209.39, 'ForeignGross': 485.32, 'Genre': 'Action', 'source': '/home/gaurav/Documents/advanced_rag/data/Cleaned_Data.csv', 'Profitability': 479.11, 'OpenProfit': 20.38, 'Story': 'Pursuit', 'Budget': 145.0, 'row': 526, 'AudienceScore': 86, 'TheatersOpenWeek': 3448, 'LeadStudio': 'Paramount', 'RottenTomatoes': 93, 'WorldGross': 694.71, 'pk': 470027294402417878}

--- Document 2 ---
Content: Movie: Rise of the Planet of the Apes
Metadata: {'BOAvgOpenWeekend': 15024, 'OpeningWeekend': 54.81, 'Year': 2011, 'DomesticGross': 176.7, 'ForeignGross': 305.04, 'Genre': 'Action', 'source': '/home/gaurav/Documents/advanced_rag/data/Cleaned_Data.csv', 'Profitability': 518.0, 'OpenProfit': 58.94, 'Story': 'Revenge', 'Budget': 93.0, 'row': 543, 'AudienceScore': 87, 'TheatersOpenWeek': 3648, 'LeadStudio': 'Fox', 'RottenTomatoes': 83, 'W